In [1]:
import pandas as pd

batches = pd.read_csv("../data/processed/batches.csv", parse_dates=["Date", "Start", "End"])
events = pd.read_csv("../data/processed/downtime_events.csv")

print("Batches:", batches.shape)
print("Downtime events:", events.shape)

Batches: (31, 11)
Downtime events: (50, 5)


In [2]:
total_batches = len(batches)
total_time = batches["Duration"].sum()
total_ideal = batches["Min batch time"].sum()
total_lost = batches["Lost Time"].sum()
efficiency = total_ideal / total_time * 100

print(f"Total batches:       {total_batches}")
print(f"Total production:    {total_time:.0f} minutes ({total_time / 60:.1f} hours)")
print(f"Total lost time:     {total_lost:.0f} minutes ({total_lost / 60:.1f} hours)")
print(f"Line efficiency:     {efficiency:.1f}%")

Total batches:       31
Total production:    3180 minutes (53.0 hours)
Total lost time:     1130 minutes (18.8 hours)
Line efficiency:     64.5%


In [3]:
by_cause = (
    events.groupby("Description")["Minutes"]
    .sum()
    .sort_values(ascending=False)
    .to_frame()
)

by_cause["% of Lost Time"] = (by_cause["Minutes"] / by_cause["Minutes"].sum() * 100).round(1)
by_cause["Cumulative %"] = by_cause["% of Lost Time"].cumsum()

by_cause

,Minutes,% of Lost Time,Cumulative %
Description,,,
Machine failure,236.0,20.9,20.9
Inventory shortage,205.0,18.1,39.0
Machine adjustment,197.0,17.4,56.4
Batch change,160.0,14.2,70.6
Batch coding error,115.0,10.2,80.8
Other,67.0,5.9,86.7
Product spill,57.0,5.0,91.7
Calibration error,34.0,3.0,94.7
Labeling error,22.0,1.9,96.6


In [4]:
by_error_type = events.groupby("Operator Error")["Minutes"].sum().to_frame()

by_error_type["% of Lost Time"] = (by_error_type["Minutes"] / by_error_type["Minutes"].sum() * 100).round(1)

by_error_type

,Minutes,% of Lost Time
Operator Error,,
No,547.0,48.4
Yes,583.0,51.6


In [5]:
by_operator = batches.groupby("Operator").agg(
    Batches=("Batch", "count"),
    Total_Lost=("Lost Time", "sum"),
    Avg_Lost_per_Batch=("Lost Time", "mean"),
    Total_Time=("Duration", "sum"),
    Ideal_Time=("Min batch time", "sum"),
)

by_operator["Efficiency %"] = (by_operator["Ideal_Time"] / by_operator["Total_Time"] * 100).round(1)
by_operator["Avg_Lost_per_Batch"] = by_operator["Avg_Lost_per_Batch"].round(1)

by_operator.sort_values("Avg_Lost_per_Batch", ascending=False)

,Batches,Total_Lost,Avg_Lost_per_Batch,Total_Time,Ideal_Time,Efficiency %
Operator,,,,,,
Mac,8,332.0,41.5,850.0,518,60.9
Dennis,5,207.0,41.4,545.0,338,62.0
Charlie,11,384.0,34.9,1158.0,774,66.8
Dee,7,207.0,29.6,627.0,420,67.0


In [6]:
events_ops = events.merge(batches[["Batch", "Operator"]], on="Batch", how="left")

op_errors = events_ops[events_ops["Operator Error"] == "Yes"]

op_error_table = op_errors.pivot_table(
    index="Operator",
    columns="Description",
    values="Minutes",
    aggfunc="sum",
    fill_value=0,
)

op_error_table["Total"] = op_error_table.sum(axis=1)
op_error_table.sort_values("Total", ascending=False)

Description,Batch change,Batch coding error,Calibration error,Label switch,Machine adjustment,Product spill,Total
Operator,,,,,,,
Charlie,10.0,44.0,24.0,10.0,118.0,22.0,228.0
Mac,130.0,47.0,0.0,0.0,15.0,0.0,192.0
Dennis,0.0,24.0,0.0,0.0,50.0,20.0,94.0
Dee,20.0,0.0,10.0,10.0,14.0,15.0,69.0


In [7]:
by_product = batches.groupby("Product").agg(
    Batches=("Batch", "count"),
    Avg_Lost_per_Batch=("Lost Time", "mean"),
    Total_Time=("Duration", "sum"),
    Ideal_Time=("Min batch time", "sum"),
)

by_product["Efficiency %"] = (by_product["Ideal_Time"] / by_product["Total_Time"] * 100).round(1)
by_product["Avg_Lost_per_Batch"] = by_product["Avg_Lost_per_Batch"].round(1)

by_product.sort_values("Efficiency %")

,Batches,Avg_Lost_per_Batch,Total_Time,Ideal_Time,Efficiency %
Product,,,,,
OR-600,1,75.0,135.0,60,44.4
CO-2L,5,55.4,767.0,490,63.9
CO-600,15,32.9,1394.0,900,64.6
DC-600,4,28.8,355.0,240,67.6
LE-600,6,28.2,529.0,360,68.1
